# 🔍 Fake News Detection — NeuroLogic '26
**Challenge 2: Fake News & Misinformation Detection**

Binary classification: `TRUE` (Real) vs `FALSE` (Fake)

Model: DistilBERT fine-tuned on title + text

In [1]:
# Install dependencies
!pip install -q transformers datasets scikit-learn

In [2]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from torch.utils.data import Dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## 1. Load & Explore Data

In [10]:
# Upload files via Colab or mount Drive
# from google.colab import files
# uploaded = files.upload()

# Or mount Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')

import csv

def read_csv_auto(path):
    """Try common encodings; handle malformed CSV rows gracefully."""
    for enc in ('utf-8-sig', 'utf-16', 'latin-1', 'cp1252'):
        try:
            df = pd.read_csv(
                path,
                encoding=enc,
                on_bad_lines='skip',   # skip rows with wrong field count
                quoting=csv.QUOTE_ALL, # treat every field as quoted
                engine='python',       # more lenient parser
            )
            print(f'{path} loaded with encoding: {enc}, shape: {df.shape}')
            return df
        except (UnicodeDecodeError, UnicodeError):
            continue
    raise ValueError(f'Could not decode {path} with any known encoding')

train_df = read_csv_auto('fakenews_with_labels.csv')
test_df  = read_csv_auto('FakeNews_no_labels.csv')

print('Train shape:', train_df.shape)
print('Test shape :', test_df.shape)
train_df.head(3)

fakenews_with_labels.csv loaded with encoding: latin-1, shape: (17794, 5)
FakeNews_no_labels.csv loaded with encoding: latin-1, shape: (1492, 5)
Train shape: (17794, 5)
Test shape : (1492, 5)


,ÿtitle,text,subject,date,label
0,VIDEO: HARLEM BAR Kicks Customers Out For Wear...,A large group of very diverse young adults who...,left-news,"May 6, 2017",False
1,PROBLEM: Trump vs. The US Intelligence Machine,21st Century Wire says The intelligence agenci...,Middle-east,"January 17, 2017",False
2,Investigators Reveal How Ex-DNC Staffer Likel...,"If you pay attention to right-wing media, and ...",News,"June 20, 2017",False


In [11]:
print('Train columns:', train_df.columns.tolist())
print('Test  columns:', test_df.columns.tolist())
print('\nFirst row:')
print(train_df.iloc[0])

# Normalize column names: strip whitespace, lowercase
train_df.columns = train_df.columns.str.strip().str.lower().str.lstrip('\ufeff').str.lstrip('ÿ')
test_df.columns  = test_df.columns.str.strip().str.lower().str.lstrip('\ufeff').str.lstrip('ÿ')

print('\nNormalized train columns:', train_df.columns.tolist())
print('\nLabel distribution:')
print(train_df['label'].value_counts())
print('\nNull values:')
print(train_df.isnull().sum())

Train columns: ['ÿtitle', 'text', 'subject', 'date', 'label']
Test  columns: ['ÿtitle', 'text', 'subject', 'date', 'label']

First row:
ÿtitle     VIDEO: HARLEM BAR Kicks Customers Out For Wear...
text       A large group of very diverse young adults who...
subject                                            left-news
date                                             May 6, 2017
label                                                  False
Name: 0, dtype: object

Normalized train columns: ['title', 'text', 'subject', 'date', 'label']

Label distribution:
label
True     10051
False     7743
Name: count, dtype: int64

Null values:
title      0
text       0
subject    0
date       0
label      0
dtype: int64


## 2. Preprocessing

In [12]:
def preprocess(df):
    df = df.copy()
    # Auto-detect title/text columns (case-insensitive)
    title_col = next((c for c in df.columns if 'title' in c), df.columns[0])
    text_col  = next((c for c in df.columns if 'text'  in c or 'content' in c), df.columns[1])
    print(f'Using title_col={title_col!r}, text_col={text_col!r}')
    df['title'] = df[title_col].fillna('')
    df['text']  = df[text_col].fillna('')
    df['input_text'] = df['title'] + ' [SEP] ' + df['text'].str[:512]
    return df

train_df = preprocess(train_df)
test_df  = preprocess(test_df)

# Encode labels — handle both 'TRUE'/'FALSE' and 'True'/'False'
train_df['label_str'] = train_df['label'].astype(str).str.strip().str.upper()
label_map = {'TRUE': 1, 'FALSE': 0}
train_df['label_id'] = train_df['label_str'].map(label_map)
print(train_df[['label', 'label_id']].value_counts())

Using title_col='title', text_col='text'
Using title_col='title', text_col='text'
label  label_id
True   1           10051
False  0            7743
Name: count, dtype: int64


## 3. Train / Validation Split

In [13]:
X_train, X_val, y_train, y_val = train_test_split(
    train_df['input_text'].tolist(),
    train_df['label_id'].tolist(),
    test_size=0.15,
    random_state=42,
    stratify=train_df['label_id']
)
print(f'Train: {len(X_train)} | Val: {len(X_val)}')

Train: 15124 | Val: 2670


## 4. Tokenization & Dataset

In [14]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
MAX_LEN    = 256  # increase to 512 if GPU memory allows

class NewsDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=MAX_LEN,
        )
        self.labels = labels

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = NewsDataset(X_train, y_train)
val_dataset   = NewsDataset(X_val,   y_val)
test_dataset  = NewsDataset(test_df['input_text'].tolist())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## 5. Model & Training

In [16]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
).to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds)}

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.000037,0.002864,0.999625
2,0.000019,0.003922,0.999625
3,0.000012,0.004104,0.999625


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=2838, training_loss=0.01179643967690982, metrics={'train_runtime': 280.0625, 'train_samples_per_second': 162.007, 'train_steps_per_second': 10.133, 'total_flos': 2382994325780208.0, 'train_loss': 0.01179643967690982, 'epoch': 3.0})

## 6. Evaluate on Validation Set

In [17]:
results = trainer.evaluate()
print(f"Validation Accuracy: {results['eval_accuracy']*100:.2f}%")

# Detailed report
val_preds = trainer.predict(val_dataset)
y_pred    = np.argmax(val_preds.predictions, axis=-1)
print('\nClassification Report:')
print(classification_report(y_val, y_pred, target_names=['Fake (FALSE)', 'Real (TRUE)']))

Validation Accuracy: 99.96%

Classification Report:
              precision    recall  f1-score   support

Fake (FALSE)       1.00      1.00      1.00      1162
 Real (TRUE)       1.00      1.00      1.00      1508

    accuracy                           1.00      2670
   macro avg       1.00      1.00      1.00      2670
weighted avg       1.00      1.00      1.00      2670



## 7. Generate Predictions on Test Set

In [ ]:
test_preds  = trainer.predict(test_dataset)
test_labels = np.argmax(test_preds.predictions, axis=-1)

id_to_label = {1: 'TRUE', 0: 'FALSE'}
submission  = test_df[['title']].copy()
submission['label'] = [id_to_label[l] for l in test_labels]

submission.to_csv('no_label.csv', index=False)
print('Saved no_label.csv')
print(submission['label'].value_counts())
submission.head()

Saved submission.csv
label
TRUE     819
FALSE    673
Name: count, dtype: int64


,title,label
0,WAKE-UP CALL! IRANIAN REFUGEE Warns The West: ...,FALSE
1,Trump Was Right: Latest Arrests Prove Threats ...,FALSE
2,BREAKING: President Trump Makes FBI Pick One D...,FALSE
3,'I can't take this any more:' Rohingya Muslims...,TRUE
4,NEWT GINGRICH HITS THE NAIL ON THE HEAD: Hereâ...,FALSE


## 8. (Optional) Save & Download Model

In [ ]:
model.save_pretrained('./fake_news_model')
tokenizer.save_pretrained('./fake_news_model')
print('Model saved.')

# Download prediction file
from google.colab import files
files.download('no_label.csv')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>